<a href="https://colab.research.google.com/github/kimanag/QM2/blob/main/BASC0005_Anastasia_Kim's_weekly_assessments_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# All weekly assessments by Anastasia Kim (me)

# Week 1
import pandas as pd

data2 = pd.read_csv("https://oballinger.github.io/BASC0005/data/assessed/w01_spotify_2023.csv")
data2

data2[['bpm', 'streams']] = data2[['streams', 'bpm']]
#need to swap the columns since it does not make sense to have so little streams and billions of bits per minute!

data2['streams'] = pd.to_numeric(data2['streams'], errors='coerce')
data2['danceability_%'] = pd.to_numeric(data2['danceability_%'], errors='coerce')

data2

billion_plus = data2[data2['streams'] > 1000000000].copy()
billion_plus.sort_values(by='danceability_%', ascending=False, inplace=True)
billion_plus

#answer is 1424589568
n=1424589568
print(n)

1424589568


In [3]:
# Week 2
import pandas as pd

flag = pd.read_csv("https://oballinger.github.io/BASC0005/data/assessed/w02_fishing_hours_by_flag_wide.csv")
# print(flag.head())

cleanflag_wide = flag.set_index("flag").replace(",", "", regex=True).astype(float).reset_index()
cleanflag_wide

cleanflag_long = cleanflag_wide.melt(id_vars="flag", var_name="month", value_name="fishing_hours")
cleanflag_long["month"] = pd.to_datetime(cleanflag_long["month"])
cleanflag_long

annual_cleanflag_long = cleanflag_long.groupby(["flag", cleanflag_long["month"].dt.year])["fishing_hours"].sum().unstack()
annual_cleanflag_long

what_we_want = annual_cleanflag_long[annual_cleanflag_long[2021] > annual_cleanflag_long[2020]]
what_we_want

print(len(what_we_want), "out of 10 flag states recorded more total fishing hours in 2021 than in 2020")


  flag   2017-01   2017-02   2017-03  2017-04  2017-05  2017-06  2017-07  \
0  CHN  516065.0  491399.0  931656.0  1374245   262149   305613   393890   
1  TWN  181201.0  157107.0  164549.0   231917   227847   210907   248714   
2  KOR    90,687    86,070   104,216  117,044  162,617  178,586  157,664   
3  ESP  152648.0  137025.0  177311.0   168193   219950   230783   238054   
4  ITA  135206.0  170771.0  205276.0   174239   273744   287198   272017   

   2017-08  2017-09  ...  2024-03  2024-04  2024-05  2024-06  2024-07  \
0  1359090  2742886  ...  4789041  5504894  1419071  1513525  1515295   
1   287478   280934  ...   346314   400427   434879   459503   468848   
2  149,919  144,196  ...  369,628  421,601  509,530  545,721  474,281   
3   241336   227040  ...   171404   217810   244080   254287   270636   
4   195548   123879  ...   173562   186129   237289   244841   245379   

   2024-08  2024-09  2024-10  2024-11  2024-12  
0  4660186  6534814  6828726  6243907  6544762  
1   50

In [4]:
# Week 3
import pandas as pd
import geopandas as gpd

eez = gpd.read_file("https://storage.googleapis.com/qm2/2627/w03/eez_galapagos.geojson")
gfw = pd.read_csv("https://oballinger.github.io/BASC0005/data/assessed/w03_gfw_effort_galapagos.csv")
# print(gfw.head())
# print(gfw.info())

gfw_corrected = gfw.copy()
gfw_corrected['lat'], gfw_corrected['lon'] = gfw['lon'], gfw['lat']
# print(gfw_corrected.head())

gfw23_per = gfw_corrected[(gfw_corrected["year"] == 2023) & (gfw_corrected["flag"] == "PER")].copy()

cells_per = gpd.GeoDataFrame(gfw23_per, geometry=gpd.points_from_xy(gfw23_per["lon"], gfw23_per["lat"]), crs="EPSG:4326")
# print (cells_per.head())

joined_per = gpd.sjoin(cells_per, eez, how="left", predicate="within")
joined_per["eez"] = joined_per["eez"].fillna("High seas")
# print(joined_per.head())

per_eez_hours = joined_per[joined_per["eez"] == "Peru"]["fishing_hours"].sum()
total_hours = joined_per["fishing_hours"].sum()

percentage = (per_eez_hours / total_hours) * 100
print(f"Inside Peru's EEZ by Peru in 2023: {percentage:.1f}%")

Inside Peru's EEZ by Peru in 2023: 73.7%
